In [520]:
import json
import re
from yarn_utils import YARNGraph
from itertools import permutations

# Load Data

In [521]:
# FOLDER_PATH = "annotations/FRACAS_12032026/"
# FILE = "105h.yarn.json"

# FOLDER_PATH = "annotations/"
# FILE = "1.yarn.json"

FOLDER_PATH = "annotations/FRACAS_1premise_yesno/"
FILE = "106h.yarn.json"

In [522]:
with open(FOLDER_PATH + FILE) as f:
    yarn_graph_json = json.load(f)

yarn_graph = YARNGraph(yarn_graph_json)
yarn_grew = yarn_graph.grew()

In [523]:
with open('output.json', 'w') as f:
    json.dump(yarn_grew, f)

In [524]:
yarn_grew

{'nodes': {'va1': {'concept': 'accountant', 'type': 'V', 'var': 'va1'},
  'vm1': {'concept': 'meeting', 'type': 'V', 'var': 'vm1'},
  'va2': {'pred': 'attend-01', 'type': 'V', 'var': 'va2'},
  's1': {'event': 's1', 'var': 's1', 'type': 'S'},
  's1-temp': {'type': 'F', 'feat': 'temp', 'var': 's1-temp'},
  's1-num': {'type': 'F', 'feat': 'num', 'var': 's1-num'},
  's1-def': {'type': 'F', 'feat': 'def', 'var': 's1-def'},
  's1-quant': {'type': 'F', 'feat': 'quant', 'var': 's1-quant'},
  's1-neg': {'type': 'F', 'feat': 'neg', 'var': 's1-neg'},
  'e1': {'rel': 'ARG0', 'type': 'E', 'var': 'e1'},
  'e2': {'rel': 'ARG1', 'type': 'E', 'var': 'e2'},
  'l1': {'value': 'past', 'var': 'l1', 'type': 'L', 'feat': 'temp'},
  'l2': {'value': 'singular', 'var': 'l2', 'type': 'L', 'feat': 'num'},
  'l3': {'value': 'singular', 'var': 'l3', 'type': 'L', 'feat': 'num'},
  'l4': {'value': 'indefinite', 'var': 'l4', 'type': 'L', 'feat': 'def'},
  'l5': {'value': 'definite', 'var': 'l5', 'type': 'L', 'feat': '

# Preprocessing

In [525]:
# from grewpy import Graph, GRS

# grs_path = "grs/main.grs"
# grs = GRS(grs_path)
# yarn_grew = grs.apply(Graph(yarn_grew), strat='main')

In [526]:
# convert temp H edges to L edges

# Build F and R

In [527]:
variables = set()
def fresh_variable(base=None):
    if base is None:
        base = 'e'
    i = 0

    while True:
        variable = base if i == 0 else f"{base}{i}"
        if variable not in variables:
            variables.add(variable)
            break
        i += 1
    return variable

id2var = {}
F = []

for node, feats in yarn_grew['nodes'].items():

    # S node quantifcation
    # includes V -> S (C edges)
    # no S -> S (D edges) for now 
    if feats['type'] == 'S':

        id2var[node] = feats['var']
        variables.add(feats['var'])

        for edge1 in yarn_grew['edges']:
            if edge1['tar'] == node:
                src = edge1['src']

                if yarn_grew['nodes'][src]['type'] == 'C':
                    for edge2 in yarn_grew['edges']:
                        if edge2['tar'] == src:
                            incoming = edge2['src']
            else:
                incoming = None

        F.append({
                        'id':node,
                        'incoming': incoming,
                        'outgoing':None,
                        'type':"∃_s",
                        'variable': id2var[node],
                        'tar_label': 'S', # change to 'label'
                    })
        
    # quantification
    if (feats['type'] == "L" or feats['type'] == 'H') and feats['feat'] in ['quant', 'temp']:
        for edge in yarn_grew['edges']:
            if edge['src'] == node:
                tar = edge['tar']

                if yarn_grew['nodes'][tar]['type'] == 'V':
                    
                    edge_label = feats['value'] if feats['value'] else feats['feat'] # accepts unlabeled temp/quant edges for now

                    if 'pred' in yarn_grew['nodes'][tar]:
                        tar_label = yarn_grew['nodes'][tar]['pred']
                    else:
                        tar_label = yarn_grew['nodes'][tar]['concept']

                    if tar not in id2var:
                        variable = fresh_variable(base=tar_label[0])
                        id2var[tar] = variable
                    else:
                        raise AssertionError(f"Double quantification. Variable for {tar} already exists in id2var.")
                    
                    F.append({
                        'id':tar,
                        'incoming':node,
                        'outgoing':None,
                        'type':"Q_"+edge_label,
                        'variable': id2var[tar] if feats['feat'] in ['quant', 'temp'] else None,
                        'tar_label':tar_label, # change to 'label'
                    })
    
    # negation, modality, aspect
    if (feats['type'] == "L" or feats['type'] == 'H') and feats['feat'] in ['neg', 'modal', 'aspect']:
        for edge1 in yarn_grew['edges']:
            if edge1['src'] == node:
                tar = edge1['tar']

                if yarn_grew['nodes'][tar]['type'] in ['V', 'L', 'H']:
                    
                    edge_label = feats['value'] if feats['value'] else feats['feat']

                    for edge2 in yarn_grew['edges']:
                        if edge2['tar'] == node:
                            src = edge2['src']

                    F.append({
                        'id':node,
                        'incoming':src,
                        'outgoing':tar,
                        'type':"Q_"+edge_label,
                        'variable': None,
                        'tar_label': None, # change to 'label'
                    })
    
    # Any none-quantified V node is a constant
for node, feats in yarn_grew['nodes'].items():
    if feats['type'] == 'V' and node not in id2var:
        if 'pred' in feats:
            id2var[node] = feats['pred'].upper()
        else:
            id2var[node] = feats['concept'].upper()
                
# Ignore definitness and number for now
#F = [f for f in F if f['type'] not in ['Q_indefinite', 'Q_singular', 'Q_plural', 'Q_manner', 'Q_perfective']]

In [528]:
id2var

{'s1': 's1', 'va2': 'a', 'vm1': 'm', 'va1': 'a1'}

In [529]:
R = {}
for node, feats in yarn_grew['nodes'].items():
    if feats['type'] == "E":
        edge_label = feats['rel']

        for edge1 in yarn_grew['edges']:
            if edge1['tar'] == node:

                src = edge1['src']

                for edge2 in yarn_grew['edges']:
                    if edge2['src'] == node:
                        tar = edge2['tar']

                        if id2var[src].isupper():
                            if tar not in R:
                                R[tar] = [(edge_label, src, tar)]
                            else:
                                R[tar].append((edge_label, src, tar))
                        else:
                            if src not in R:
                                R[src] = [(edge_label, src, tar)]
                            else:
                                R[src].append((edge_label, src, tar))
    
    if feats['type'] == "L" and feats['feat'] == 'num' and feats['value'] == 'plural': 
        for edge1 in yarn_grew['edges']:
            if edge1['src'] == node:
                tar = edge1['tar']

                if yarn_grew['nodes'][tar]['type'] == 'V':
                    
                    if tar not in R:
                        R[tar] = [('plural', tar)] 
                    else:
                        R[tar].append(('plural', tar))
    
    if feats['type'] == "L" and feats['feat'] == 'def': 
        for edge1 in yarn_grew['edges']:
            if edge1['src'] == node:
                tar = edge1['tar']

                if yarn_grew['nodes'][tar]['type'] == 'V':
                    
                    if tar not in R:
                        R[tar] = [('C', tar)]
                    else:
                        R[tar].append(('C', tar))

    if feats['type'] == "C":
        edge_label = feats['rel']

        for edge1 in yarn_grew['edges']:
            if edge1['tar'] == node:
                src = edge1['src']

                for edge2 in yarn_grew['edges']:
                    if edge2['src'] == node:
                        tar = edge2['tar']
                
                        if tar not in R:
                            R[tar] = [(edge_label, src, tar)]
                        else:
                            R[tar].append((edge_label, src, tar))

    # S node modifiers
    # update as you annotate if you find trickier cases
        
    if feats['type'] == "L" and feats['feat'] in ['manner', 'loc', 'dir', 'duration', 'mod', 'freq']: 
        for edge1 in yarn_grew['edges']:
            if edge1['src'] == node:
                tar = edge1['tar']

                if yarn_grew['nodes'][tar]['type'] == 'V':
                    
                    edge_label = feats['value'] if feats['value'] else feats['feat']

                    if 'pred' in yarn_grew['nodes'][tar]:
                        tar_label = yarn_grew['nodes'][tar]['pred']
                    else:
                        tar_label = yarn_grew['nodes'][tar]['concept']

                    for edge2 in yarn_grew['edges']:
                        if edge2['tar'] == node:
                            src = edge2['src'].split('-')[0]
                        
                            if src not in R:
                                R[src] = [(edge_label, src, tar)]
                            else:
                                R[src].append((edge_label, src, tar))

In [530]:
F

[{'id': 's1',
  'incoming': None,
  'outgoing': None,
  'type': '∃_s',
  'variable': 's1',
  'tar_label': 'S'},
 {'id': 'va2',
  'incoming': 'l1',
  'outgoing': None,
  'type': 'Q_past',
  'variable': 'a',
  'tar_label': 'attend-01'},
 {'id': 'vm1',
  'incoming': 'l7',
  'outgoing': None,
  'type': 'Q_exists',
  'variable': 'm',
  'tar_label': 'meeting'},
 {'id': 'va1',
  'incoming': 'h1',
  'outgoing': None,
  'type': 'Q_exists',
  'variable': 'a1',
  'tar_label': 'accountant'},
 {'id': 'h2',
  'incoming': 's1-neg',
  'outgoing': 'h1',
  'type': 'Q_neg',
  'variable': None,
  'tar_label': None}]

In [531]:
R

{'va2': [('ARG0', 'va2', 'va1'), ('ARG1', 'va2', 'vm1')],
 'va1': [('C', 'va1')],
 'vm1': [('C', 'vm1')]}

# Create the Forest

In [ ]:
forest = {'nodes':{}, 'edges':[]}

for i, f in enumerate(F):
    forest['nodes'][i] = {
                            'id': f['id'],
                            'incoming': f['incoming'],
                            'outgoing': f['outgoing'],
                            'type': f['type'],
                            'variable': f['variable'],
                            'tar_label': f['tar_label'],
                            'relations': [
                                f"{rel[0]}({id2var[rel[1]]},{id2var[rel[2]]})" if len(rel) == 3
                                else f"{rel[0]}({id2var[rel[1]]})"
                                for rel in R[f['id']]
                            ] if f['id'] in R else [],
                        }


for k1, v1 in forest['nodes'].items(): # encode specified scope
    for k2, v2 in forest['nodes'].items():
        if v1['id'] == v2['incoming']:
            forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})
        if v1['id'] == v2['outgoing']:
            forest['edges'].append({'src':k2, 'rel':'', 'tar':k1})
        if v1['outgoing'] and v2['incoming'] and v1['outgoing'] == v2['incoming']: # not sure this is smart
            forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})

In [565]:
forest

{'nodes': {0: {'id': 's1',
   'incoming': None,
   'outgoing': None,
   'type': '∃_s',
   'variable': 's1',
   'tar_label': 'S',
   'relations': []},
  1: {'id': 'va2',
   'incoming': 'l1',
   'outgoing': None,
   'type': 'Q_past',
   'variable': 'a',
   'tar_label': 'attend-01',
   'relations': ['ARG0(a,a1)', 'ARG1(a,m)']},
  2: {'id': 'vm1',
   'incoming': 'l7',
   'outgoing': None,
   'type': 'Q_exists',
   'variable': 'm',
   'tar_label': 'meeting',
   'relations': ['C(m)']},
  3: {'id': 'va1',
   'incoming': 'h1',
   'outgoing': None,
   'type': 'Q_exists',
   'variable': 'a1',
   'tar_label': 'accountant',
   'relations': ['C(a1)']},
  4: {'id': 'h2',
   'incoming': 's1-neg',
   'outgoing': 'h1',
   'type': 'Q_neg',
   'variable': None,
   'tar_label': None,
   'relations': []}},
 'edges': [{'src': 4, 'rel': '', 'tar': 3}]}

## Add the Participant before Event constraint

In [566]:
R

{'va2': [('ARG0', 'va2', 'va1'), ('ARG1', 'va2', 'vm1')],
 'va1': [('C', 'va1')],
 'vm1': [('C', 'vm1')]}

In [567]:
# Predicates are introduced after their arguments (E relations only)
# C relations are encoded already in the Forest building forest

forest = forest.copy()
for _, rels in R.items():
    for rel in rels:
        if len(rel) == 3:
            src = rel[1]
            tar = rel[2]

        for k1, v1 in forest['nodes'].items():
            for k2, v2 in forest['nodes'].items():
                if v1['id'] == src and v2['id'] == tar and \
                    yarn_grew['nodes'][v1['id']]['type'] == 'V' and \
                    yarn_grew['nodes'][v2['id']]['type'] == 'V':
                    
                    forest['edges'].append({'src':k2, 'rel':'', 'tar':k1})

In [568]:
def get_S_descendants(graph):
    nodes = graph["nodes"]
    edges = graph["edges"]

    adj = {}
    for e in edges:
        adj.setdefault(e["src"], []).append(e["tar"])

    result = {}

    for node_id, node_data in nodes.items():
        if node_data.get("type") != "S":
            continue

        visited = set()
        stack = [node_id]
        reachable_V = set()

        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)

            current_type = nodes[current].get("type")

            if current_type in ["L", "H", "V"]:
                reachable_V.add(current)

            if current_type == "C":
                continue

            for neighbor in adj.get(current, []):
                if neighbor not in visited:
                    stack.append(neighbor)

        result[node_id] = list(reachable_V)

    return result

In [569]:
s_descendants = get_S_descendants(yarn_grew)
for k1, v1 in forest['nodes'].items():
    if v1['id'] in s_descendants:
        for k2, v2 in forest['nodes'].items():
            if v2['id'] in s_descendants[v1['id']]:
                forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})

# Get All Possible Trees

In [570]:
import itertools
import networkx as nx

def get_all_possible_trees(n):
    nodes = list(range(n))
    for seq in itertools.product(nodes, repeat=n-2):
        yield nx.from_prufer_sequence(seq)

In [571]:
nodes = list(forest['nodes'].keys())
n_nodes = len(nodes)

all_possible_trees = []

for tree in get_all_possible_trees(n_nodes):

    for root in nodes:

        visited = set([root])
        stack = [root]
        directed_edges = []

        while stack:
            current = stack.pop()

            for neighbor in tree.neighbors(current):
                if neighbor not in visited:
                    visited.add(neighbor)
                    stack.append(neighbor)

                    directed_edges.append({'src': current,'tar': neighbor})

        all_possible_trees.append({'edges': directed_edges})

In [572]:
all_possible_trees

[{'edges': [{'src': 0, 'tar': 1},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 3},
   {'src': 0, 'tar': 4}]},
 {'edges': [{'src': 1, 'tar': 0},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 3},
   {'src': 0, 'tar': 4}]},
 {'edges': [{'src': 2, 'tar': 0},
   {'src': 0, 'tar': 1},
   {'src': 0, 'tar': 3},
   {'src': 0, 'tar': 4}]},
 {'edges': [{'src': 3, 'tar': 0},
   {'src': 0, 'tar': 1},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 4}]},
 {'edges': [{'src': 4, 'tar': 0},
   {'src': 0, 'tar': 1},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 3}]},
 {'edges': [{'src': 0, 'tar': 2},
   {'src': 0, 'tar': 3},
   {'src': 0, 'tar': 1},
   {'src': 1, 'tar': 4}]},
 {'edges': [{'src': 1, 'tar': 0},
   {'src': 1, 'tar': 4},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 3}]},
 {'edges': [{'src': 2, 'tar': 0},
   {'src': 0, 'tar': 3},
   {'src': 0, 'tar': 1},
   {'src': 1, 'tar': 4}]},
 {'edges': [{'src': 3, 'tar': 0},
   {'src': 0, 'tar': 2},
   {'src': 0, 'tar': 1},
   {'src': 1, 'tar': 4}]},
 

In [573]:
len(all_possible_trees)

625

# Build T_all

In [574]:
# Gets the children of nodes that don't introduce variables
def get_H_children(graph):
    H_children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        if not graph['nodes'][src]['variable']:
            H_children_dict[src] = H_children_dict.get(src, []) + [tar]

    return H_children_dict

In [575]:
# Get all children
def get_children(graph):
    children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        children_dict[src] = children_dict.get(src, []) + [tar]
    return children_dict

# Extend children to descendants
def get_descendants(node, children_dict):
    descendants = []
    for child in children_dict.get(node, []):
        descendants.append(child)
        descendants.extend(get_descendants(child, children_dict))

    return descendants

def get_all_descendants(graph):
    children_dict = get_children(graph)
    descendants_dict = {}
    for node in children_dict:
        descendants_dict[node] = get_descendants(node, children_dict)

    return descendants_dict

In [576]:
# Constraint Checkers

def check_compatibility_of_scopes(tree, forest):
    descendants_tree = get_all_descendants(tree)
    descendants_forest = get_all_descendants(forest)

    for k,v in descendants_forest.items():
        for descendant in v:
            if k in descendants_tree:
                if descendant not in descendants_tree[k]:
                    return False
            else:
                return False
    return True

def check_locality_of_features(tree, forest):
    children_tree = get_children(tree)
    H_children_forest = get_H_children(forest)
    
    for k,v in H_children_forest.items():
        for child in v:
            if k in children_tree:
                if child not in children_tree[k]:
                    return False
            else:
                return False
    return True

In [577]:
descendants_forest = get_all_descendants(forest)
H_children_forest = get_H_children(forest)
print(descendants_forest)
print(H_children_forest)

{4: [3, 1], 3: [1], 2: [1, 1, 1], 0: [1, 2, 1, 1, 1, 3, 1, 4, 3, 1]}
{4: [3]}


In [578]:
valid_trees = [tree for tree in all_possible_trees if check_locality_of_features(tree, forest)]
valid_trees = [tree for tree in valid_trees if check_compatibility_of_scopes(tree, forest)]
valid_trees

[{'edges': [{'src': 0, 'tar': 2},
   {'src': 2, 'tar': 4},
   {'src': 4, 'tar': 3},
   {'src': 3, 'tar': 1}]},
 {'edges': [{'src': 0, 'tar': 4},
   {'src': 4, 'tar': 3},
   {'src': 3, 'tar': 2},
   {'src': 2, 'tar': 1}]}]

In [579]:
T_all = []
for tree in valid_trees:
    new_tree = forest.copy()
    new_tree['edges'] = tree['edges']
    T_all.append(new_tree)

In [580]:
T_all

[{'nodes': {0: {'id': 's1',
    'incoming': None,
    'outgoing': None,
    'type': '∃_s',
    'variable': 's1',
    'tar_label': 'S',
    'relations': []},
   1: {'id': 'va2',
    'incoming': 'l1',
    'outgoing': None,
    'type': 'Q_past',
    'variable': 'a',
    'tar_label': 'attend-01',
    'relations': ['ARG0(a,a1)', 'ARG1(a,m)']},
   2: {'id': 'vm1',
    'incoming': 'l7',
    'outgoing': None,
    'type': 'Q_exists',
    'variable': 'm',
    'tar_label': 'meeting',
    'relations': ['C(m)']},
   3: {'id': 'va1',
    'incoming': 'h1',
    'outgoing': None,
    'type': 'Q_exists',
    'variable': 'a1',
    'tar_label': 'accountant',
    'relations': ['C(a1)']},
   4: {'id': 'h2',
    'incoming': 's1-neg',
    'outgoing': 'h1',
    'type': 'Q_neg',
    'variable': None,
    'tar_label': None,
    'relations': []}},
  'edges': [{'src': 0, 'tar': 2},
   {'src': 2, 'tar': 4},
   {'src': 4, 'tar': 3},
   {'src': 3, 'tar': 1}]},
 {'nodes': {0: {'id': 's1',
    'incoming': None,
    'ou

## Reformat T_all

In [581]:
# Reformat to linear tree for easier interpretation

def graph_to_linear_tree(graph):
    nodes = graph["nodes"]
    edges = graph["edges"]

    next_node = {}
    for e in edges:
        next_node[e["src"]] = e["tar"]

    all_nodes = set(nodes.keys())
    all_targets = {e["tar"] for e in edges}
    root = (all_nodes - all_targets).pop()

    def build(node_id):
        node_data = dict(nodes[node_id])

        if node_id in next_node:
            node_data["child"] = build(next_node[node_id])
        else:
            node_data["child"] = None

        return node_data

    return build(root)

In [582]:
T_all = [graph_to_linear_tree(tree) for tree in T_all]

In [583]:
T_all[0]

{'id': 's1',
 'incoming': None,
 'outgoing': None,
 'type': '∃_s',
 'variable': 's1',
 'tar_label': 'S',
 'relations': [],
 'child': {'id': 'vm1',
  'incoming': 'l7',
  'outgoing': None,
  'type': 'Q_exists',
  'variable': 'm',
  'tar_label': 'meeting',
  'relations': ['C(m)'],
  'child': {'id': 'h2',
   'incoming': 's1-neg',
   'outgoing': 'h1',
   'type': 'Q_neg',
   'variable': None,
   'tar_label': None,
   'relations': [],
   'child': {'id': 'va1',
    'incoming': 'h1',
    'outgoing': None,
    'type': 'Q_exists',
    'variable': 'a1',
    'tar_label': 'accountant',
    'relations': ['C(a1)'],
    'child': {'id': 'va2',
     'incoming': 'l1',
     'outgoing': None,
     'type': 'Q_past',
     'variable': 'a',
     'tar_label': 'attend-01',
     'relations': ['ARG0(a,a1)', 'ARG1(a,m)'],
     'child': None}}}}}

# Interpretation

In [584]:
def conj(parts):
    parts = [p for p in parts if p and p.strip()]
    return " ∧ ".join(parts)


def wrap_quant(q, var, head, relations, body, connective="∧"):
    """
    Builds:
    Qx. ( head(x) ∧ {rels} {connective} {body} )
    """

    rel = conj(relations)
    head_part = f"{head}({var})"

    if rel:
        left = f"{head_part} ∧ {rel}"
    else:
        left = head_part

    return f"{q}{var}. ( {left}\n {connective} ({body}) )"

def clean_formula(formula):
    clean_formula = formula.replace(" ∧ ()", "")
    return clean_formula

In [585]:
def interpret(root, temp_variable):
    
    if root is None:
        return ""
    
    if root["type"] == "∃_s" or root["type"] == "Q_exists":
        return wrap_quant(
            "∃",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret(root["child"], temp_variable),
            connective="∧"
        )
    
    if root["type"] == "Q_forall":
        return wrap_quant(
            "∀",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret(root["child"], temp_variable),
            connective="→"
        )
    
    if root["type"] == "Q_present":
        temp = f"{root['variable']}_O_{temp_variable}"
        return wrap_quant(
                    "∃",
                    root["variable"],
                    root["tar_label"],
                    root["relations"] + [temp],
                    interpret(root["child"], temp_variable),
                    connective="∧"
        )

    if root["type"] == "Q_past":
        temp = f"{root['variable']}≺{temp_variable}"
        return wrap_quant(
                    "∃",
                    root["variable"],
                    root["tar_label"],
                    root["relations"] + [temp],
                    interpret(root["child"], temp_variable),
                    connective="∧"
        )
    
    if root["type"] == "Q_future":
        temp = f"{temp_variable}≺{root['variable']}"
        return wrap_quant(
                    "∃",
                    root["variable"],
                    root["tar_label"],
                    root["relations"] + [temp],
                    interpret(root["child"], temp_variable),
                    connective="∧"
        )
    
    if root["type"] == "Q_neg":
        return f"¬( {interpret(root['child'], temp_variable)} )"
    
    if root["type"] == "Q_possibility":
        return f"◇( {interpret(root['child'], temp_variable)} )"

    if root["type"] == "Q_necessity":
        return f"□( {interpret(root['child'], temp_variable)} )"

In [586]:
for tree in T_all:
    print(clean_formula(interpret(tree, 'NOW')))
    print("---")

∃s1. ( S(s1)
 ∧ (∃m. ( meeting(m) ∧ C(m)
 ∧ (¬( ∃a1. ( accountant(a1) ∧ C(a1)
 ∧ (∃a. ( attend-01(a) ∧ ARG0(a,a1) ∧ ARG1(a,m) ∧ a≺NOW
 )) ) )) )) )
---
∃s1. ( S(s1)
 ∧ (¬( ∃a1. ( accountant(a1) ∧ C(a1)
 ∧ (∃m. ( meeting(m) ∧ C(m)
 ∧ (∃a. ( attend-01(a) ∧ ARG0(a,a1) ∧ ARG1(a,m) ∧ a≺NOW
 )) )) ) )) )
---


# Vampire

In [557]:
def conj_tptp(parts):
    parts = [p for p in parts if p and p.strip()]
    return " & ".join(parts)

def wrap_quant_tptp(q, var, head, relations, body, connective):
    rel = conj_tptp(relations)
    head_part = f"{head}({var})"

    if rel:
        left = f"{head_part} & {rel}"
    else:
        left = head_part

    return f"{q} [{var}] : ( {left} {connective} ( {body} ) )"

def clean_formula_tptp(formula):
    clean_formula = formula.replace(" & (  )", "").replace("-", "_")
    return clean_formula

In [558]:
def interpret_vampire(root, temp_variable):
    
    if root is None:
        return ""
    
    if root["type"] == "∃_s" or root["type"] == "Q_exists":
        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )

    if root["type"] == "Q_forall":
        return wrap_quant_tptp(
            "!",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret_vampire(root["child"], temp_variable),
            "=>"
        )

    if root["type"] == "Q_present":
        temp = f"{root['variable']}_O_{temp_variable}"

        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"] + [temp],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )

    if root["type"] == "Q_past":
        temp = f"before({root['variable']},{temp_variable})"

        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"] + [temp],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )
    
    if root["type"] == "Q_future":
        temp = f"before({temp_variable},{root['variable']})"

        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"] + [temp],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )
    
    if root["type"] == "Q_neg":
        return f"~( {interpret_vampire(root['child'], temp_variable)} )"

In [559]:
print(clean_formula_tptp(interpret_vampire(T_all[0], "NOW")))

? [s1] : ( S(s1) & ( ~(  ) ) )
